# S11 — BI/ML distribuido con Spark
## Proyecto propio: Bitcoin (BTC/USDT)

**Estudiante:** Julmer Quispe Apaza  
**Curso:** Big Data  
**Docente:** Abel Angel Sullon Macalupu  
**Proyecto:** Predicción del precio de Bitcoin BTC/USDT con Spark, Kafka, Prometheus y Grafana  
**Repositorio:** https://github.com/julmerqa/bigdata

## Objetivo
Continuar el trabajo de S10 con dos bloques:

1. **BI batch:** calcular KPIs sobre las predicciones BTC/USDT ya persistidas.
2. **ML en vivo:** recargar el `PipelineModel` de S10, aplicarlo nuevamente al topic `crypto-btc-eventos` y publicar cada micro-lote en Pushgateway para verlo en Prometheus y Grafana.


## 0. Prerrequisitos

Este notebook debe colocarse en:

`pyspark/sesiones/s11-bi-ml-distribuido/`

Deben estar disponibles:
- Docker Desktop.
- `lambda26-pyspark`.
- `lambda26-kafka`.
- `lambda26-kafka-ui`.
- `lambda26-uso-crypto`.
- El modelo S10 `modelo_btc_lr`.
- El Parquet S10 `predicciones_stream_btc`.
- Prometheus, Grafana y Pushgateway en `lambda26-kafka-net`.

### Agregar a `obs/compose.yml`
```yaml
pushgateway:
  image: prom/pushgateway:v1.11.1
  container_name: lambda26-pushgateway
  restart: unless-stopped
  ports:
    - "49091:9091"
  networks:
    - lambda26-kafka-net
```

### Agregar a `obs/prometheus/prometheus.yml`
```yaml
- job_name: pushgateway
  honor_labels: true
  metrics_path: /metrics
  static_configs:
    - targets:
        - pushgateway:9091
```

Luego:
```powershell
cd D:\BD\lambda26\obs
docker compose up -d
docker ps
```

Servicios:
- Pushgateway: `http://localhost:49091`
- Prometheus: `http://localhost:49090`
- Grafana: `http://localhost:43000`


## 1. SparkSession

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("S11-Crypto-BTC-BI-ML")
    .master("local[*]")
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.13:4.2.0")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")
spark


:: loading settings :: url = jar:file:/usr/local/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2.5.2/cache
The jars for the packages stored in: /root/.ivy2.5.2/jars
org.apache.spark#spark-sql-kafka-0-10_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-7dcec64d-3c73-4b75-afd3-91361e549107;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.13;4.2.0 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.13;4.2.0 in central
	found org.apache.kafka#kafka-clients;3.9.2 in central
	found at.yawk.lz4#lz4-java;1.11.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.8 in central
	found org.slf4j#slf4j-api;2.0.17 in central
	found org.apache.hadoop#hadoop-client-runtime;3.5.0 in central
	found org.apache.hadoop#hadoop-client-api;3.5.0 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.scala-lang.mod

## 2. Rutas del proyecto

In [2]:
RUTA_MODELO_S10 = "../s10-series-tiempo-inferencia-streaming/artifacts/models/modelo_btc_lr"
RUTA_PREDICCIONES_S10 = "../s10-series-tiempo-inferencia-streaming/artifacts/predicciones_stream_btc"

RUTA_PREDICCIONES_S11 = "./artifacts/predicciones_live_btc"
CHECKPOINT_S11 = "./artifacts/chk_predicciones_live_btc"

TOPIC_BTC = "crypto-btc-eventos"

PUSHGATEWAY_BASE_URL = "http://pushgateway:9091"
PROMETHEUS_BASE_URL = "http://prometheus:9090"

print("Modelo S10:", RUTA_MODELO_S10)
print("Predicciones S10:", RUTA_PREDICCIONES_S10)
print("Topic:", TOPIC_BTC)


Modelo S10: ../s10-series-tiempo-inferencia-streaming/artifacts/models/modelo_btc_lr
Predicciones S10: ../s10-series-tiempo-inferencia-streaming/artifacts/predicciones_stream_btc
Topic: crypto-btc-eventos


# PARTE A — BI batch sobre evidencia propia de S10

Se reutiliza el Parquet real generado por tu proyecto BTC en S10.

KPIs:
- total de predicciones comparables;
- MAE en USDT;
- MAPE porcentual;
- ratio de predicciones con error porcentual ≤ 0.10 %.


## 3. Cargar el Parquet real de S10

In [3]:
predicciones_s10 = spark.read.parquet(RUTA_PREDICCIONES_S10)

print(f"Filas persistidas en S10: {predicciones_s10.count():,}")
predicciones_s10.printSchema()
predicciones_s10.orderBy("fechaMercado").show(10, truncate=False)


Filas persistidas en S10: 29
root
 |-- batchId: integer (nullable = true)
 |-- symbol: string (nullable = true)
 |-- fechaRecepcion: timestamp (nullable = true)
 |-- fechaMercado: timestamp (nullable = true)
 |-- close: double (nullable = true)
 |-- prediccionSiguienteClose: double (nullable = true)



[Stage 4:>                                                          (0 + 9) / 9]

+-------+-------+-----------------------+-------------------+--------+------------------------+
|batchId|symbol |fechaRecepcion         |fechaMercado       |close   |prediccionSiguienteClose|
+-------+-------+-----------------------+-------------------+--------+------------------------+
|0      |BTCUSDT|2026-10-21 16:22:07.942|2024-12-01 03:25:00|96049.23|96048.91683395056       |
|1      |BTCUSDT|2026-10-21 16:22:10.945|2024-12-01 03:26:00|96022.63|96022.64958386729       |
|1      |BTCUSDT|2026-10-21 16:22:13.95 |2024-12-01 03:27:00|96015.94|96015.88958328798       |
|1      |BTCUSDT|2026-10-21 16:22:16.954|2024-12-01 03:28:00|96044.01|96043.78297260692       |
|1      |BTCUSDT|2026-10-21 16:22:19.959|2024-12-01 03:29:00|96047.61|96047.67077550966       |
|1      |BTCUSDT|2026-10-21 16:22:22.963|2024-12-01 03:30:00|96037.43|96036.51934589584       |
|1      |BTCUSDT|2026-10-21 16:22:25.968|2024-12-01 03:31:00|96109.01|96107.62205671794       |
|1      |BTCUSDT|2026-10-21 16:22:28.971

## 4. Reconstruir el valor real del minuto siguiente

In [4]:
from pyspark.sql.window import Window

ventana_btc = Window.orderBy("fechaMercado")

predicciones_unicas = (
    predicciones_s10
    .dropDuplicates(["fechaMercado"])
    .orderBy("fechaMercado")
)

comparado_s10 = (
    predicciones_unicas
    .withColumn("closeRealSiguiente", F.lead("close", 1).over(ventana_btc))
    .withColumn(
        "errorAbsoluto",
        F.abs(F.col("prediccionSiguienteClose") - F.col("closeRealSiguiente"))
    )
    .withColumn(
        "errorPct",
        (F.col("errorAbsoluto") / F.col("closeRealSiguiente")) * 100
    )
    .na.drop(subset=["closeRealSiguiente"])
)

comparado_s10.select(
    "fechaMercado",
    "close",
    "prediccionSiguienteClose",
    "closeRealSiguiente",
    "errorAbsoluto",
    "errorPct"
).show(20, truncate=False)


+-------------------+--------+------------------------+------------------+------------------+---------------------+
|fechaMercado       |close   |prediccionSiguienteClose|closeRealSiguiente|errorAbsoluto     |errorPct             |
+-------------------+--------+------------------------+------------------+------------------+---------------------+
|2024-12-01 03:25:00|96049.23|96048.91683395056       |96022.63          |26.286833950551227|0.02737566545568605  |
|2024-12-01 03:26:00|96022.63|96022.64958386729       |96015.94          |6.709583867283072 |0.006987989564319291 |
|2024-12-01 03:27:00|96015.94|96015.88958328798       |96044.01          |28.120416712015867|0.02927867829760114  |
|2024-12-01 03:28:00|96044.01|96043.78297260692       |96047.61          |3.8270273930829717|0.00398451079947015  |
|2024-12-01 03:29:00|96047.61|96047.67077550966       |96037.43          |10.240775509664672|0.010663316906402714 |
|2024-12-01 03:30:00|96037.43|96036.51934589584       |96109.01         

## 5. KPIs de negocio BTC

In [5]:
kpi_btc = comparado_s10.agg(
    F.count("*").alias("totalPredicciones"),
    F.avg("errorAbsoluto").alias("maeUsdt"),
    F.avg("errorPct").alias("mapePct"),
    F.avg((F.col("errorPct") <= F.lit(0.10)).cast("int")).alias("ratioDentro01Pct")
).first()

print("KPI BTC — evidencia real S10")
print("---------------------------------------------")
print(f"Predicciones comparables : {kpi_btc['totalPredicciones']}")
print(f"MAE                      : {kpi_btc['maeUsdt']:.6f} USDT")
print(f"MAPE                     : {kpi_btc['mapePct']:.6f} %")
print(f"Ratio error <= 0.10 %    : {kpi_btc['ratioDentro01Pct']:.4f}")


KPI BTC — evidencia real S10
---------------------------------------------
Predicciones comparables : 28
MAE                      : 21.654855 USDT
MAPE                     : 0.022499 %
Ratio error <= 0.10 %    : 1.0000


## 6. Publicar los KPIs batch en Pushgateway

In [6]:
import urllib.request

metricas_btc = {
    "lambda26_crypto_bi_predicciones_total": float(kpi_btc["totalPredicciones"]),
    "lambda26_crypto_bi_mae_usdt": float(kpi_btc["maeUsdt"]),
    "lambda26_crypto_bi_mape_pct": float(kpi_btc["mapePct"]),
    "lambda26_crypto_bi_aciertos_01pct_ratio": float(kpi_btc["ratioDentro01Pct"]),
}

lineas = []
for nombre, valor in metricas_btc.items():
    lineas.append(f"# TYPE {nombre} gauge")
    lineas.append(f"{nombre} {valor}")

cuerpo = "\n".join(lineas) + "\n"

PUSHGATEWAY_KPI_URL = (
    PUSHGATEWAY_BASE_URL
    + "/metrics/job/lambda26_crypto_bi/instance/s11_btc_batch"
)

peticion = urllib.request.Request(
    PUSHGATEWAY_KPI_URL,
    data=cuerpo.encode("utf-8"),
    method="PUT"
)

with urllib.request.urlopen(peticion) as respuesta:
    print("Pushgateway respondió:", respuesta.status)

print("✓ KPIs batch BTC publicados correctamente")


Pushgateway respondió: 200
✓ KPIs batch BTC publicados correctamente


## 7. Verificar los KPIs en Prometheus

In [7]:
import json
import urllib.parse

def consultar_prometheus(nombre_metrica):
    query = urllib.parse.quote(nombre_metrica, safe="")
    url = f"{PROMETHEUS_BASE_URL}/api/v1/query?query={query}"
    resultado = json.load(urllib.request.urlopen(url))
    return resultado["data"]["result"]

for nombre in metricas_btc:
    valores = consultar_prometheus(nombre)
    if valores:
        print(f"{nombre}: {valores[0]['value'][1]}")
    else:
        print(f"{nombre}: SIN DATOS (espera un scrape_interval)")


lambda26_crypto_bi_predicciones_total: SIN DATOS (espera un scrape_interval)
lambda26_crypto_bi_mae_usdt: SIN DATOS (espera un scrape_interval)
lambda26_crypto_bi_mape_pct: SIN DATOS (espera un scrape_interval)
lambda26_crypto_bi_aciertos_01pct_ratio: SIN DATOS (espera un scrape_interval)


# PARTE B — ML en vivo

Aquí **no se reentrena** el modelo.  
Se recarga el mismo `PipelineModel` ganador de S10 y se aplica al stream BTC/USDT.


## 8. Recargar el PipelineModel de S10

In [8]:
from pyspark.ml import PipelineModel

modelo_cargado = PipelineModel.load(RUTA_MODELO_S10)

print("✓ PipelineModel cargado")
print("Ruta:", RUTA_MODELO_S10)


✓ PipelineModel cargado
Ruta: ../s10-series-tiempo-inferencia-streaming/artifacts/models/modelo_btc_lr


## 9. Esquema del evento BTC

In [9]:
from pyspark.sql.types import StructType, StringType, DoubleType, LongType

esquema_evento_btc = (
    StructType()
    .add("tipoEvento", StringType())
    .add("symbol", StringType())
    .add("timestamp", LongType())
    .add("marketOpenTime", LongType())
    .add("open", DoubleType())
    .add("high", DoubleType())
    .add("low", DoubleType())
    .add("close", DoubleType())
    .add("volume", DoubleType())
    .add("quote_volume", DoubleType())
    .add("trades", LongType())
    .add("taker_buy_volume", DoubleType())
    .add("origen", StringType())
)

print("✓ Esquema BTC creado")


✓ Esquema BTC creado


## 10. Leer `crypto-btc-eventos` desde Kafka

In [10]:
crudo_btc = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "kafka:9092")
    .option("subscribe", TOPIC_BTC)
    .option("startingOffsets", "latest")
    .option("failOnDataLoss", "false")
    .load()
)

eventos_btc = (
    crudo_btc
    .select(F.col("value").cast("string").alias("json_str"))
    .select(F.from_json(F.col("json_str"), esquema_evento_btc).alias("e"))
    .select("e.*")
)

print("✓ Conectado a:", TOPIC_BTC)
print("¿Es streaming?:", eventos_btc.isStreaming)
eventos_btc.printSchema()


✓ Conectado a: crypto-btc-eventos
¿Es streaming?: True
root
 |-- tipoEvento: string (nullable = true)
 |-- symbol: string (nullable = true)
 |-- timestamp: long (nullable = true)
 |-- marketOpenTime: long (nullable = true)
 |-- open: double (nullable = true)
 |-- high: double (nullable = true)
 |-- low: double (nullable = true)
 |-- close: double (nullable = true)
 |-- volume: double (nullable = true)
 |-- quote_volume: double (nullable = true)
 |-- trades: long (nullable = true)
 |-- taker_buy_volume: double (nullable = true)
 |-- origen: string (nullable = true)



## 11. Aplicar el modelo sin reentrenar

In [11]:
predicciones_stream = modelo_cargado.transform(eventos_btc)

print("✓ PipelineModel aplicado al stream BTC")
print("¿Predicciones streaming?:", predicciones_stream.isStreaming)


✓ PipelineModel aplicado al stream BTC
¿Predicciones streaming?: True


## 12. Funciones de publicación por micro-lote

In [12]:
PUSHGATEWAY_STREAM_BASE = (
    PUSHGATEWAY_BASE_URL
    + "/metrics/job/lambda26_crypto_ml_streaming/instance"
)

def publicar_prediccion_btc(symbol, close_real, prediccion):
    cuerpo = (
        "# TYPE lambda26_crypto_btc_close_real gauge\n"
        f"lambda26_crypto_btc_close_real {close_real}\n"
        "# TYPE lambda26_crypto_btc_prediccion_siguiente gauge\n"
        f"lambda26_crypto_btc_prediccion_siguiente {prediccion}\n"
    )

    peticion = urllib.request.Request(
        f"{PUSHGATEWAY_STREAM_BASE}/{symbol}",
        data=cuerpo.encode("utf-8"),
        method="PUT"
    )
    urllib.request.urlopen(peticion).close()


def procesar_microlote_btc(df_microlote, id_lote):
    filas = df_microlote.count()

    if filas == 0:
        print(f"Micro-lote {id_lote}: sin filas")
        return

    salida = df_microlote.select(
        F.lit(id_lote).alias("batchId"),
        "symbol",
        (F.col("timestamp") / 1000).cast("timestamp").alias("fechaRecepcion"),
        (F.col("marketOpenTime") / 1000).cast("timestamp").alias("fechaMercado"),
        "close",
        F.col("prediction").alias("prediccionSiguienteClose")
    )

    salida.write.mode("append").parquet(RUTA_PREDICCIONES_S11)

    for fila in df_microlote.select("symbol", "close", "prediction").collect():
        publicar_prediccion_btc(
            fila["symbol"],
            fila["close"],
            fila["prediction"]
        )

    print(
        f"Micro-lote {id_lote}: {filas} predicciones BTC "
        "guardadas y publicadas en vivo"
    )

print("✓ foreachBatch preparado")


✓ foreachBatch preparado


## 13. Iniciar el productor antes del streaming

En PowerShell:
```powershell
cd D:\BD\lambda26\uso-crypto
docker compose up -d
docker exec -it lambda26-uso-crypto python producer_crypto.py
```

Déjalo corriendo.


In [13]:
consulta_btc = (
    predicciones_stream
    .writeStream
    .foreachBatch(procesar_microlote_btc)
    .option("checkpointLocation", CHECKPOINT_S11)
    .start()
)

print("✓ Streaming S11 BTC iniciado")
print("Consulta activa:", consulta_btc.isActive)


✓ Streaming S11 BTC iniciado
Consulta activa: True


## 14. Revisar progreso

In [14]:
print("Consulta activa:", consulta_btc.isActive)
print("Último progreso:")
print(consulta_btc.lastProgress)


Consulta activa: True
Último progreso:
None
Micro-lote 0: sin filas


netlib-blas: JNI_OnLoad: dlopen(libblas.so.3) failed: libblas.so.3: cannot open shared object file: No such file or directory
                                                                                

## 15. Confirmar que la métrica cambia en Prometheus

In [15]:
import time

def valor_prometheus(nombre_metrica):
    valores = consultar_prometheus(nombre_metrica)
    if not valores:
        return None
    return valores[0]["value"][1]

metrica_live = "lambda26_crypto_btc_close_real"

primera = valor_prometheus(metrica_live)
print("Primera lectura:", primera)

print("Esperando 20 segundos...")
time.sleep(20)

segunda = valor_prometheus(metrica_live)
print("Segunda lectura:", segunda)

if primera is not None and segunda is not None and primera != segunda:
    print("✓ Confirmado: la métrica cambió; está en vivo.")
elif primera == segunda and primera is not None:
    print("⚠ El valor fue igual. Espera otra lectura y vuelve a ejecutar.")
else:
    print("⚠ Prometheus todavía no devuelve datos.")


Primera lectura: None
Esperando 20 segundos...


Micro-lote 1: 1 predicciones BTC guardadas y publicadas en vivo


Micro-lote 2: 1 predicciones BTC guardadas y publicadas en vivo
Micro-lote 3: 1 predicciones BTC guardadas y publicadas en vivo


Micro-lote 4: 1 predicciones BTC guardadas y publicadas en vivo


Micro-lote 5: 1 predicciones BTC guardadas y publicadas en vivo
Micro-lote 6: 1 predicciones BTC guardadas y publicadas en vivo


...

Micro-lote 605: 1 predicciones BTC guardadas y publicadas en vivo


Micro-lote 606: 1 predicciones BTC guardadas y publicadas en vivo


Micro-lote 607: 1 predicciones BTC guardadas y publicadas en vivo


Micro-lote 608: 1 predicciones BTC guardadas y publicadas en vivo


Micro-lote 609: 1 predicciones BTC guardadas y publicadas en vivo


# 16. Grafana

Abre `http://localhost:43000` y crea:

## Paneles batch
- `lambda26_crypto_bi_predicciones_total` → **Stat**
- `lambda26_crypto_bi_mae_usdt` → **Stat**
- `lambda26_crypto_bi_aciertos_01pct_ratio` → **Stat**, Unit: Percent (0.0-1.0)

## Panel en vivo
En el mismo panel **Time series**:
- `lambda26_crypto_btc_close_real`
- `lambda26_crypto_btc_prediccion_siguiente`

Configuración:
- rango: **Last 5 minutes**
- refresh: **5s**
- leyendas: `BTC real` y `Predicción siguiente`


## 17. Detener el stream cuando ya tengas las capturas

In [16]:
consulta_btc.stop()

print("✓ Streaming S11 BTC detenido")
print("Consulta activa:", consulta_btc.isActive)


✓ Streaming S11 BTC detenido
Consulta activa: False


## 18. Auditoría post-hoc opcional

In [17]:
predicciones_s11 = spark.read.parquet(RUTA_PREDICCIONES_S11)

predicciones_s11_unicas = (
    predicciones_s11
    .dropDuplicates(["fechaMercado"])
    .orderBy("fechaMercado")
)

ventana_s11 = Window.orderBy("fechaMercado")

auditoria_s11 = (
    predicciones_s11_unicas
    .withColumn("closeRealSiguiente", F.lead("close", 1).over(ventana_s11))
    .withColumn(
        "errorAbsoluto",
        F.abs(F.col("prediccionSiguienteClose") - F.col("closeRealSiguiente"))
    )
    .na.drop(subset=["closeRealSiguiente"])
)

auditoria_s11.select(
    "fechaMercado",
    "close",
    "prediccionSiguienteClose",
    "closeRealSiguiente",
    "errorAbsoluto"
).show(20, truncate=False)

kpi_produccion = auditoria_s11.agg(
    F.count("*").alias("totalComparado"),
    F.avg("errorAbsoluto").alias("maeProduccion")
).first()

print("Predicciones comparables:", kpi_produccion["totalComparado"])
print(f"MAE en producción: {kpi_produccion['maeProduccion']:.6f} USDT")


+-------------------+--------+------------------------+------------------+------------------+
|fechaMercado       |close   |prediccionSiguienteClose|closeRealSiguiente|errorAbsoluto     |
+-------------------+--------+------------------------+------------------+------------------+
|2024-12-01 00:22:00|96565.05|96565.24093342916       |96544.0           |21.24093342915876 |
|2024-12-01 00:23:00|96544.0 |96544.66291469024       |96555.15          |10.487085309752729|
|2024-12-01 00:24:00|96555.15|96555.1730795265        |96608.0           |52.82692047349701 |
|2024-12-01 00:25:00|96608.0 |96606.82575683595       |96548.95          |57.87575683595787 |
|2024-12-01 00:26:00|96548.95|96549.3851998567        |96584.0           |34.614800143303   |
|2024-12-01 00:27:00|96584.0 |96583.82035407217       |96604.0           |20.17964592782664 |
|2024-12-01 00:28:00|96604.0 |96603.74183422401       |96601.41          |2.3318342240090715|
|2024-12-01 00:29:00|96601.41|96601.05895225132       |96616

[Stage 3096:=============================================>        (17 + 3) / 20]

Predicciones comparables: 608
MAE en producción: 25.349853 USDT


## 19. Limpiar la métrica live del Pushgateway

In [ ]:
symbol = "BTCUSDT"

peticion_borrar = urllib.request.Request(
    f"{PUSHGATEWAY_STREAM_BASE}/{symbol}",
    method="DELETE"
)

urllib.request.urlopen(peticion_borrar).close()

print("✓ Métricas live BTC eliminadas del Pushgateway")


# 20. Error o hallazgo técnico

Pushgateway conserva el último valor publicado aunque el productor o el stream ya se hayan detenido. Por ello, un panel de Grafana puede parecer activo cuando en realidad solo está mostrando un valor congelado. La verificación correcta consiste en comprobar que la misma consulta PromQL devuelve valores distintos en momentos separados y limpiar el grupo del Pushgateway al finalizar.


# 21. Reflexión técnica

El Pushgateway es apropiado para la parte batch porque el notebook calcula los KPIs, los publica una vez y termina. En cambio, el modelo BTC que funciona continuamente podría exponer un endpoint `/metrics` y permitir que Prometheus lo consulte directamente mediante `pull`, que es el patrón recomendado para servicios de larga duración. En esta práctica se utiliza Pushgateway como un atajo para simplificar la infraestructura. Esto se relaciona con la Regla #37 de Google: no basta con medir el modelo antes del despliegue, sino que se debe observar su comportamiento mientras recibe datos reales. En esta implementación el modelo se redespliega sobre el stream BTC/USDT y su salida se observa en Prometheus y Grafana mientras ocurre.


# 22. Evidencias para el informe

Captura:
1. KPIs batch calculados.
2. Pushgateway con respuesta 200.
3. KPIs consultados desde Prometheus.
4. PipelineModel S10 cargado.
5. Conexión a `crypto-btc-eventos`.
6. Micro-lotes publicados en vivo.
7. Dos consultas PromQL con valores diferentes.
8. Dashboard Grafana con paneles batch.
9. Dashboard Grafana con BTC real vs predicción.
10. Hallazgo técnico y reflexión.


# 23. Git

```powershell
cd D:\BD\lambda26

git status

git add pyspark/sesiones/s11-bi-ml-distribuido/11_bi_ml_distribuido_crypto_btc.ipynb

git commit -m "feat(s11): publicar KPIs y predicciones BTC en BI ML"

git push
```

## Arquitectura final

**Datos BTC → Spark ML → PipelineModel S10 → Kafka → Spark Structured Streaming → Pushgateway → Prometheus → Grafana**
